Gated Recurrent Unit (GRU) — Character-Level Text Generation

In [2]:
# CELL 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import requests

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
# CELL 2 — Load text data (Tiny Shakespeare — classic char-level RNN/GRU benchmark)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
print(f"Total characters: {len(text)}")
print(text[:300])

Total characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [4]:
# CELL 3 — Build character vocabulary and encode text
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encoded_text = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)
print(f"Vocab size: {vocab_size}")
print("Sample encoding:", encoded_text[:20])

Vocab size: 65
Sample encoding: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]


In [5]:
# CELL 4 — Create sequence dataset
SEQ_LEN = 100

def make_sequences(data, seq_len):
    inputs, targets = [], []
    for i in range(0, len(data) - seq_len - 1, seq_len):
        inputs.append(data[i:i + seq_len])
        targets.append(data[i + 1:i + seq_len + 1])  # next-character prediction
    return np.array(inputs), np.array(targets)

# Use a subset for manageable training time
subset = encoded_text[:200000]
X, Y = make_sequences(subset, SEQ_LEN)

X_t = torch.tensor(X, dtype=torch.long)
Y_t = torch.tensor(Y, dtype=torch.long)

from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X_t, Y_t)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Number of sequences: {len(dataset)}")

Number of sequences: 1999


In [6]:
# CELL 5 — Model definition
class CharGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # nn.GRU uses tanh for candidate hidden state and sigmoid for update/reset gates —
        # this is the standard GRU internal design, not swapped out
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)                    # [batch, seq_len, embed_dim]
        output, hidden = self.gru(embedded, hidden)      # output: [batch, seq_len, hidden_dim]
        logits = self.fc(output)                         # [batch, seq_len, vocab_size]
        return logits, hidden

model = CharGRU(vocab_size=vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)
print(model)

CharGRU(
  (embedding): Embedding(65, 128)
  (gru): GRU(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=256, out_features=65, bias=True)
)


In [7]:
# CELL 6 — Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits, _ = model(batch_x)  # hidden state reset each batch (stateless training)
        loss = criterion(logits.reshape(-1, vocab_size), batch_y.reshape(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # standard for RNN-family training
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(loader)
    perplexity = np.exp(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f} | Perplexity: {perplexity:.2f}")

Epoch 1/5 — Loss: 2.8566 | Perplexity: 17.40
Epoch 2/5 — Loss: 2.1718 | Perplexity: 8.77
Epoch 3/5 — Loss: 1.9228 | Perplexity: 6.84
Epoch 4/5 — Loss: 1.7619 | Perplexity: 5.82
Epoch 5/5 — Loss: 1.6630 | Perplexity: 5.28


In [8]:
# CELL 7 — Text generation
def generate_text(model, start_str, length=300, temperature=0.8):
    model.eval()
    chars_generated = list(start_str)
    input_seq = torch.tensor([[char_to_idx[ch] for ch in start_str]], dtype=torch.long).to(device)
    hidden = None

    with torch.no_grad():
        # Warm up hidden state on the seed string
        logits, hidden = model(input_seq, hidden)

        last_char_idx = input_seq[0, -1].unsqueeze(0).unsqueeze(0)
        for _ in range(length):
            logits, hidden = model(last_char_idx, hidden)
            logits = logits[0, -1] / temperature
            probs = torch.softmax(logits, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            chars_generated.append(idx_to_char[next_idx])
            last_char_idx = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(chars_generated)

generated = generate_text(model, start_str="ROMEO:", length=300, temperature=0.8)
print(generated)

ROMEO:
I folced for of serve the iven: and dost or the forther
Hath our chimes my denion.

Third Servingman:
He now ham ald fell by himely know,
But third how even your spear for to
Thy should so sto not?

CORIOLANUS:
Here out dissing.

Second Rery,
What him my hoin with again a and not
bided same, let th


In [11]:
# CELL 8 — Compare generation at different temperatures
for temp in [0.3, 0.8, 1.2, 1.5, 2.0]:
    print(f"--- Temperature: {temp} ---")
    print(generate_text(model, start_str="ROMEO:", length=150, temperature=temp))
    print()

--- Temperature: 0.3 ---
ROMEO:
What the country shall the consul he hall
The more the gods and here and shall be stand and the report the hang the contern of his bear the have and 

--- Temperature: 0.8 ---
ROMEO:
My sunder'd in thild see
Do that are well that cansurous of this sword and wlaths, lielt.
We sice to speak and in the must out;
And that he knond bef

--- Temperature: 1.2 ---
ROMEO: yet as Murder,
We had the ended to be most.

Firsce, add is
I jut pluckier, do sike lick help.
GLernt Citizem,
Hothind art his penein man's our I
And

--- Temperature: 1.5 ---
ROMEO: when no Viderorsay.
Were out yes!
Deakmy,-hasd, filjMoth spemg!
Gom he hespI in'd-pleqby: follius to?

IUPGBRALEHKTEEs Se;
Af phose? lorl,, larysu ap

--- Temperature: 2.0 ---
ROMEO:'ges?

CokNC;GCitizegisy Nano-onkugeKAmad yu worcoone trikar:r-L a Rwzo:r
viticlmy, sajiut ca'no wefuecridnian:
Ey,,
Inlifjole'in:; Befdietboritaxif-o

